Okay! so we (you, prof. Silva, and I) met and tried to sort my societies into continents and it did not work :) so in the notes of checkpoint 2 i said i would find an offline way, and i did! i downloaded the geopandas library *(offine, unlike reverse geocoder)* and used gemini to get it working

In [3]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import geopandas as gpd
from shapely.geometry import Point
from geodatasets import get_url

import pandas as pd
import numpy as np

In [4]:
societies = pd.read_csv('../D-PLACE-dplace-cldf-908f302/cldf/societies.csv')

societies.head()

,ID,Name,Latitude,Longitude,Glottocode,Name_and_ID_in_source,xd_id,alt_names_by_society,main_focal_year,HRAF_name_ID,HRAF_ID,origLat,origLong,comment,glottocode_comment,region,type,Language_Level_Glottocodes,ISO639P3code,Contribution_ID
0,B72,!Kung,-20.00,21.18,juho1239,!Kung (B72),xd1,Kung Bushmen; !Kung (Was Nyae); Was Nyae,1950.0,San (FX10),FX10,-20.00,21.18,Original,NaN,Southern Africa,society,juho1239,NaN,dplace-dataset-binford
1,B296,Kuskowagmut,61.01,-161.55,kusk1241,Kuskowagmut (B296),xd1000,NaN,1850.0,NaN,NaN,61.01,-161.55,Original,NaN,Subarctic America,society,cent2127,NaN,dplace-dataset-binford
2,B322,Syilx,49.46,-119.63,sout2963,Okanagan (B322),xd1001,Okanagan,1860.0,NaN,NaN,49.46,-119.63,Original,NaN,Western Canada,society,okan1243,NaN,dplace-dataset-binford
3,B335,Round Lake Ojibwa,52.71,-90.62,seve1241,Round Lake Ojibwa (B335),xd1003,Weagamow; North Caribou Lake; Round Lake,1900.0,NaN,NaN,52.71,-90.62,Original,NaN,Eastern Canada,society,seve1240,NaN,dplace-dataset-binford
4,B339,North Albany Ojibwa,51.22,-83.10,wini1244,North Albany Ojibwa (B339),xd1004,North Albany,1850.0,NaN,NaN,51.22,-83.10,Original,NaN,Eastern Canada,society,seve1240,NaN,dplace-dataset-binford


In [5]:
#start gemini

# 2. Convert to a GeoDataFrame
# Replace 'lon' and 'lat' with your actual column names
geoData = gpd.GeoDataFrame(
    societies, 
    geometry=gpd.points_from_xy(societies['Longitude'], societies['Latitude']),
    crs="EPSG:4326"
)

In [6]:
# 3. Load the world map directly from the web
# Using the 110m resolution "admin_0_countries" map
world_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(world_url)

# 4. Filter to just what we need to keep it fast
world = world[['CONTINENT', 'geometry']]

In [7]:
world.tail()

,CONTINENT,geometry
172,Europe,"POLYGON ((18.82982 45.90887, 18.82984 45.90888..."
173,Europe,"POLYGON ((20.0707 42.58863, 19.80161 42.50009,..."
174,Europe,"POLYGON ((20.59025 41.85541, 20.52295 42.21787..."
175,North America,"POLYGON ((-61.68 10.76, -61.105 10.89, -60.895..."
176,Africa,"POLYGON ((30.83385 3.50917, 29.9535 4.1737, 29..."


In [8]:
# 5. Spatial Join
# 'predicate="within"' checks if the point is inside the continent polygon
result = gpd.sjoin(geoData, world, how="left", predicate="within")
# End Gemini

In [9]:
result.head()

,ID,Name,Latitude,Longitude,Glottocode,Name_and_ID_in_source,xd_id,alt_names_by_society,main_focal_year,HRAF_name_ID,...,comment,glottocode_comment,region,type,Language_Level_Glottocodes,ISO639P3code,Contribution_ID,geometry,index_right,CONTINENT
0,B72,!Kung,-20.00,21.18,juho1239,!Kung (B72),xd1,Kung Bushmen; !Kung (Was Nyae); Was Nyae,1950.0,San (FX10),...,Original,NaN,Southern Africa,society,juho1239,NaN,dplace-dataset-binford,POINT (21.18 -20),49.0,Africa
1,B296,Kuskowagmut,61.01,-161.55,kusk1241,Kuskowagmut (B296),xd1000,NaN,1850.0,NaN,...,Original,NaN,Subarctic America,society,cent2127,NaN,dplace-dataset-binford,POINT (-161.55 61.01),4.0,North America
2,B322,Syilx,49.46,-119.63,sout2963,Okanagan (B322),xd1001,Okanagan,1860.0,NaN,...,Original,NaN,Western Canada,society,okan1243,NaN,dplace-dataset-binford,POINT (-119.63 49.46),3.0,North America
3,B335,Round Lake Ojibwa,52.71,-90.62,seve1241,Round Lake Ojibwa (B335),xd1003,Weagamow; North Caribou Lake; Round Lake,1900.0,NaN,...,Original,NaN,Eastern Canada,society,seve1240,NaN,dplace-dataset-binford,POINT (-90.62 52.71),3.0,North America
4,B339,North Albany Ojibwa,51.22,-83.10,wini1244,North Albany Ojibwa (B339),xd1004,North Albany,1850.0,NaN,...,Original,NaN,Eastern Canada,society,seve1240,NaN,dplace-dataset-binford,POINT (-83.1 51.22),3.0,North America


In [10]:
#just to see if there are any other "types"\
type_group = result.groupby('type').count()['ID']
type_group

type
languoid    4085
society     2599
Name: ID, dtype: int64

In [11]:
# I'm going to just focus on socieities
result_soc = result[result['type'] == 'society']
result_soc.shape

(2599, 23)

In [12]:
#Let's make an easier dataframe
SocDF = result_soc.drop(columns=['Glottocode', 'xd_id', 'HRAF_name_ID', 'comment', 'glottocode_comment', 'Language_Level_Glottocodes', 'ISO639P3code', 'Contribution_ID', 'type', 'HRAF_ID', 'origLat', 'origLong', 'geometry', 'index_right'])
#i removed 13 columns that I either didn't need/want or were redundant
SocDF.head()

,ID,Name,Latitude,Longitude,Name_and_ID_in_source,alt_names_by_society,main_focal_year,region,CONTINENT
0,B72,!Kung,-20.00,21.18,!Kung (B72),Kung Bushmen; !Kung (Was Nyae); Was Nyae,1950.0,Southern Africa,Africa
1,B296,Kuskowagmut,61.01,-161.55,Kuskowagmut (B296),NaN,1850.0,Subarctic America,North America
2,B322,Syilx,49.46,-119.63,Okanagan (B322),Okanagan,1860.0,Western Canada,North America
3,B335,Round Lake Ojibwa,52.71,-90.62,Round Lake Ojibwa (B335),Weagamow; North Caribou Lake; Round Lake,1900.0,Eastern Canada,North America
4,B339,North Albany Ojibwa,51.22,-83.10,North Albany Ojibwa (B339),North Albany,1850.0,Eastern Canada,North America


In [13]:
SocDF['main_focal_year'] = pd.to_numeric(SocDF['main_focal_year'])

# Society DataFrame Cleaned Up :D
### Now that we can organize traditions via continent, I'm going to import some more datasets

**The plan is**
Agregate datasets concerning:
* different cultures/traditions, geographically and chronologically 

In [14]:
continent = SocDF.groupby('CONTINENT').count()['ID']
continent

CONTINENT
Africa           709
Asia             337
Europe           155
North America    815
Oceania          155
South America    222
Name: ID, dtype: int64

In [15]:
fig1 = go.Figure(data=[go.Pie(
    labels = continent.index,
    values=continent.values,
    title="Traditions by Continent"
    )])

fig1.show()
#Include info here about the disparity in research from in different continents, as well as the development of writing, etc.
#Figure out how to use vector shapes to use a map in plotly

In [16]:
year_group = SocDF.groupby('main_focal_year').count()['ID']

year_group

main_focal_year
-6500.0    1
-2500.0    1
-2000.0    1
-1750.0    1
-1400.0    1
          ..
 1978.0    2
 1979.0    1
 1980.0    3
 1989.0    2
 1990.0    1
Name: ID, Length: 137, dtype: int64

In [17]:
# scatter timeline
fig2 = go.Figure(data=[go.Scatter(x=year_group.index,
                    y=year_group.values,
                    mode='markers'
                    )])

fig2.show()

# Folklore Database

I'm constrained by what CSVs I can export, which is to say, only results from searches. My plan right now is to show an
**overview of common themes** *(hero's journey, shapeshifting, infidelity, warnings, eternal punishment/tasks)*
- timeline (line chart)
- continents (bubbles?)

**Overview on Epics** (i'd love to compare the attributes of global tradition's heroes)
**the most common themes today** (https://www.readomania.com/blog/popular-literary-trends-of-2025)

# World Folk Epics

### "World folk-epics are those epics which are not just literary masterpieces but also an integral part of the worldview of a people. They were originally oral literatures, which were later written down by either single author or several writers."

from https://en.wikipedia.org/wiki/List_of_world_folk-epics but also UNESCO's list of *Intangible Cultural Heritage of Humanity*.

My list of caveats:

    • This is inherently qualitative data shown quantitatively, in order to give it a new context. What it isn't is "accurate". I cannot find every single myth or tale with moral warnings, eternal tasks, or whatever else. I am only one design and technology student, and even scholars who dedicate their lives to this research would know that a comprehensive and accurate depiction of the global oral tradition of humanity is impossible.
    
    • I used the earliest dates I could find with evidentiary support. Many of these from 1500+ CE are much older, but scholars have not agreed upon even loose dates for their oral creation.
    
    • Timelines for oral tradition are inherently difficult because 1) Word of mouth does not create a "paper trail", and 2) Many stories have edits and addendums to make them more pertinent for the times, which means parts of the stories may have references to events thousands of years ago, whereas others could have only been added in the semi recent past.


In [18]:
# worldepic = pd.read_csv('world-folk-epics.csv')
#after looking at plotly's geo graphs I added long/lat (with gemini)

worldepic = pd.read_csv('world-folk-epics-longlat.csv')

worldepic.head()

,Title,Description,Language,Tradition,Region,Year (Rough),Lat,Long
0,T'heydinn,"a Mauritanian epic ensemble, performed in Has...",Hassaniya,Mauritanian,African,1600.0,20.0,-12.0
1,Gassire's Lute,"an epic from the Soninke people, West Africa",Soninke,Soninke,African,400.0,15.0,-8.0
2,Bayajidda,a West African epic,Hausa,Hausa,African,900.0,12.5,7.5
3,Eri,a West African epic,Igbo,Igbo,African,900.0,6.0,7.0
4,Oduduwa,a West African epic,Yoruba,Yoruba,African,1000.0,7.5,4.5


In [19]:
epicdata = pd.DataFrame(worldepic)

epicdata.head()

,Title,Description,Language,Tradition,Region,Year (Rough),Lat,Long
0,T'heydinn,"a Mauritanian epic ensemble, performed in Has...",Hassaniya,Mauritanian,African,1600.0,20.0,-12.0
1,Gassire's Lute,"an epic from the Soninke people, West Africa",Soninke,Soninke,African,400.0,15.0,-8.0
2,Bayajidda,a West African epic,Hausa,Hausa,African,900.0,12.5,7.5
3,Eri,a West African epic,Igbo,Igbo,African,900.0,6.0,7.0
4,Oduduwa,a West African epic,Yoruba,Yoruba,African,1000.0,7.5,4.5


Cool, time to think about how to visualize this...

I could go for a world map, or maybe a map timeline? that could be interesting

In [20]:
traces = go.Scattergeo(
                    lat=epicdata["Lat"],
                    lon=epicdata["Long"],
                    text= epicdata["Title"],
                    customdata= epicdata["Description"])
fig3 = go.Figure(data=[traces]) 
fig3.update_geos(projection_type="natural earth")
fig3.update_layout(height=300, margin={"r":0,"t":0,"l":0,"b":0})

fig3.update_traces(
    hovertemplate="<b>%{text}</b> "\
    "<br>%{customdata}",
    mode='markers',
    marker=dict(size=10, opacity=.5)
)

fig3.show()

In [21]:
#Great now I want to make it look nicer, colors and whatnot
layout = go.Layout(height=300, margin={"r":0,"t":0,"l":0,"b":0})

fig3 = go.Figure(layout=layout) 

# Loop through each unique Region to create separate traces
for r in epicdata['Region'].unique():
    epicdata_sub = epicdata[epicdata['Region'] == r]
    fig3.add_trace(go.Scattergeo(
                lat=epicdata_sub["Lat"],
                lon=epicdata_sub["Long"],
                text= epicdata_sub["Title"],
                customdata= epicdata_sub["Description"],
                name=r
    ))

fig3.update_geos(projection_type="natural earth")

fig3.update_traces(
    hovertemplate="<b>%{text}</b> "\
    "<br>%{customdata}",
    mode='markers',
    marker=dict(size=10, opacity=.5)
)
#this is my reminder to add titles for the traces and the map lol
fig3.show()

In [22]:
layout = go.Layout(title="World Folk Epics by Tradition", legend_title="Region")
                #    , height=300, margin={"r":0,"t":0,"l":0,"b":0})

fig3 = go.Figure(layout=layout) 

# Loop through each unique Region to create separate traces
for r in epicdata['Region'].unique():
    epicdata_sub = epicdata[epicdata['Region'] == r]
    fig3.add_trace(go.Scattergeo(
                lat=epicdata_sub["Lat"],
                lon=epicdata_sub["Long"],
                text= epicdata_sub["Title"],
                customdata= epicdata_sub["Description"],
                name=r
    ))

fig3.update_geos(projection_type="natural earth")

fig3.update_traces(
    hovertemplate="<b>%{text}</b> "\
    "<br>%{customdata}",
    mode='markers',
    marker=dict(size=10, opacity=.5)
)
fig3.show()

# Now I want to try something new... a timeline

I **think** it should be straightforward from the documentation https://plotly.com/python/animations/ I basically just need to nest my for loop that makes regions so then it can check for the time and then split the regions... if that makes sense. It does to me, so lets roll with it.

In [23]:
for year in epicdata["Year (Rough)"]:
    frame = {"data": []}

I changed my mind, that would have sucked and thinking about it I'm not even convinced it would show the data better.
https://plotly.com/python/mixed-subplots/

I like the idea of multiple subplots all interconnected to a storyline, lets make our globe with world epics then show maybe a timeline and a bar or pie chart to show disparity of data in different regions?

In [24]:
epicdata["Year_nm"] = pd.to_numeric(epicdata["Year (Rough)"])

epicdata["Year_nm"] = pd.to_datetime(epicdata["Year_nm"], format='%Y')

epicdata.head()
#well that's odd
#and now that i've researched it won't work anyway because pandas time can't be negative

OutOfBoundsDatetime: Out of bounds nanosecond timestamp: 1600, at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [25]:
type(epicdata["Year (Rough)"][0])
#okay so it is a float right now... that seems fine

numpy.float64

In [26]:
years = sorted(epicdata['Year (Rough)'].unique())

In [27]:
# Create figure
fig4 = go.Figure()

# Add traces, one for each slider step
for step in np.arange(0, 5, 0.5):
    for r in epicdata['Region'].unique():
        epicdata_sub = epicdata[epicdata['Region'] == r]
        fig3.add_trace(go.Scattergeo(
                    lat=epicdata_sub["Lat"],
                    lon=epicdata_sub["Long"],
                    text= epicdata_sub["Title"],
                    customdata= epicdata_sub["Description"],
                    name=r
        ))

# Make 10th trace visible
fig4.data[10].visible = True

# Create and add slider
steps = []
for i in range(len(fig4.data)):
    step = dict(
        method="update",
        args=[{"visible": [False] * len(fig4.data)},
              {"title": "Slider switched to step: " + str(i)}],  # layout attribute
    )
    step["args"][0]["visible"][i] = True  # Toggle i'th trace to "visible"
    steps.append(step)

sliders = [dict(
    active=10,
    currentvalue={"prefix": "Frequency: "},
    pad={"t": 50},
    steps=steps
)]

fig4.update_layout(
    sliders=sliders
)

fig4.show()

IndexError: tuple index out of range

In [ ]:
# I was suffering w the timeline aspect so I gave my code to Gemini
fig = go.Figure()

frame_data = []
# 1. Add Initial Traces (The starting point of the animation)
first_year = years[0]
for region in epicdata['Region'].unique():
    # Show only data from the very first available year
    mask = (epicdata['Region'] == region) & (epicdata['Year (Rough)'] <= first_year)
    sub = epicdata[mask]
    
    fig.add_trace(go.Scattergeo(
        lat=sub["Lat"],
        lon=sub["Long"],
        text=sub["Title"],
        customdata=sub["Description"],
        name=region,
        hovertemplate="<b>%{text}</b><br>%{customdata}",
        mode='markers',
        marker=dict(size=10, opacity=.5)
    ))

# 2. Create the Frames (One for each 'step' in time)
frames = []
for yr in years:
    frame_data = []
    for region in epicdata['Region'].unique():
        # Cumulative: Show everything up to the current slider year
        mask = (epicdata['Region'] == region) & (epicdata['Year (Rough)'] <= yr)
        sub = epicdata[mask]
        
        frame_data.append(go.Scattergeo(
            lat=sub["Lat"],
        lon=sub["Long"],
        text=sub["Title"],
        customdata=sub["Description"],
        name=region,
        hovertemplate="<b>%{text}</b> "\
            "<br>%{customdata}",
        mode='markers',
        marker=dict(size=10, opacity=.5)
        ))
    frames.append(go.Frame(data=frame_data, name=str(yr)))

fig.frames = frames

fig.update_layout(
    title="The Spread of Global Epics Over Time",
    showlegend=True,
    geo=dict(
        projection_type='natural earth',
        # showland=True,
        # landcolor="rgb(243, 243, 243)",
    ),
    updatemenus=[{
        "type": "buttons",
        "buttons": [{
            "label": "Play",
            "method": "animate",
            "args": [None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}]
        }]
    }],
    sliders=[{
        "active": 0,
        "steps": [{
            "label": str(yr),
            "method": "animate",
            "args": [[str(yr)], {
                "frame": {"duration": 100, "redraw": True}, 
                "fromcurrent": True,
                "transition": {"duration": 500},
                "mode": "immediate"}]
        } for yr in years]
    }]
)

fig.show()